# Project-out results: Loss Δ vs layer and vs k

Two plots:
1. **Layer (x) vs Loss Δ (y)** for a **specific k** — use when you have one result file per layer (e.g. runs with `--output-dir`). Title shows k.
2. **k (x) vs Loss Δ (y)** for a **specific layer** — use one result file that has multiple k (e.g. `--top-k "1,4,16,64,128"`). Title shows layer.

**Why does a single run in the terminal look different from the "layer vs delta" plot?**
- The **terminal** shows one run = **one layer** (one point). The **plot** shows many layers (one point per layer). So the terminal output is a single point on that plot; which layer you ran (`--layer_idx`) determines where it sits (e.g. early layers often have larger SVD deltas, later layers can be smaller).
- **Dataset and baseline** matter: e.g. `wiki_titles` (baseline ~7–8) vs `wikitext` (baseline ~2.5) change the scale of loss and deltas. Old plots may be from a different dataset or code version.
- To **match** the layer plot with current code: run for **many layers** (e.g. `for layer in 0 5 10 15 20 25; do ... --layer_idx $layer --output-dir results ...`) and plot; your single-run deltas will then appear as one layer’s point on the curve.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments.router_interventions import (
    load_results,
    plot_delta_vs_layers,
    plot_delta_vs_k,
)

# Default folder for result JSONs (used by both plots)
RESULTS_DIR = ROOT / "results"

## 1. Plot: Loss Δ vs layer (for a specific k)

Load all result JSONs from a folder (one file per layer, each with `by_k`). Pick **k** for the title and x = layer index, y = loss delta.

In [ ]:
# Override if your results are elsewhere
RESULTS_DIR = RESULTS_DIR  # or Path("/path/to/your/results")

# Which k to use for this plot (must exist in each file's by_k)
K_FOR_LAYER_PLOT = 4

# Load all project_out_*.json in the folder
result_files = sorted(RESULTS_DIR.glob("project_out_*.json"))
if not result_files:
    raise FileNotFoundError(f'No project_out_*.json in {RESULTS_DIR}. Run with --output-dir {RESULTS_DIR} for multiple layers.')

results_by_layer = [load_results(p) for p in result_files]
print(f"Loaded {len(results_by_layer)} result files. Layers: {[r.get('config', {}).get('layer_idx') for r in results_by_layer]}")

In [ ]:
%matplotlib inline

fig = plot_delta_vs_layers(
    results_by_layer,
    K_FOR_LAYER_PLOT,
    title=f"Loss Δ vs layer (k={K_FOR_LAYER_PLOT})",
)
fig.show()

## 2. Plot: Loss Δ vs k (for a specific layer)

Load **one** result file that has multiple k (e.g. from a run with `--top-k "1,4,16,64,128"`). x = k, y = loss delta. Title shows layer.

In [ ]:
# One file with by_k (multiple k values) for one layer. Set to your file or use a glob.
RESULTS_FILE_ONE_LAYER = RESULTS_DIR / "project_out_L13_k1-4-16-64-128_wiki_titles_q8bit.json"  # example
if not RESULTS_FILE_ONE_LAYER.exists():
    candidates = list(RESULTS_DIR.glob("project_out_*.json"))
    RESULTS_FILE_ONE_LAYER = candidates[0] if candidates else RESULTS_FILE_ONE_LAYER

results_one_layer = load_results(RESULTS_FILE_ONE_LAYER)
layer_idx = results_one_layer.get("config", {}).get("layer_idx", "?")
print(f"Layer: {layer_idx}, by_k keys: {list(results_one_layer.get('by_k', {}).keys())}")

In [ ]:
fig2 = plot_delta_vs_k(
    results_one_layer,
    title=f"Loss Δ vs k (layer {layer_idx})",
)
fig2.show()